# 05j: Create an Experiment Report

Turn a reproducible small experiment into a reviewable report: Markdown, numeric JSON/CSV, and a confusion-matrix PNG. Run every cell in order; this notebook trains its own model and does not depend on results from another lab.

**Prerequisites:** Python 3.10+, NumPy, pandas, scikit-learn and Matplotlib in the active kernel; see [setup](README.md). No downloads, gateway, GPU or credentials. Allow about 15 minutes. Outputs go to the ignored `05_Jupyter/lab_outputs/experiment_report/` directory. Reruns overwrite only this lab's four named outputs.

**Goal:** make the provenance, split, parameters, limitations and evidence inspectable, rather than exporting a single impressive accuracy number.

## Install the notebook dependencies if needed

The next cell installs the shared CPU packages into **this notebook's selected Python kernel** using `%pip`. It includes **scikit-learn**, whose import name is `sklearn`, plus NumPy, pandas, matplotlib, Pillow, and IPython.

If the packages are missing or this is a fresh environment, set `INSTALL_PACKAGES = True` and run the cell once. Only do this when installation is permitted and pip is configured to use your approved package source. If the environment is already prepared, leave it `False`. The cell finds `requirements.txt` whether the kernel starts in the repository root or `05_Jupyter`.

After a successful installation, change the flag back to `False`, use **Kernel → Restart Kernel**, and run the notebook from the top. A restart clears old imports and variables. If installation fails or the source is blocked, stop here and ask the administrator to prepare the environment or approved mirror; do not bypass TLS checks.

PyTorch/CUDA, interactive widgets, ClearML, and live gateway extras remain optional. See the [support guide](README.md#prepare-one-approved-python-environment) before enabling them. Package installation can use the network; the lesson's default dataset path remains offline.


In [ ]:
from pathlib import Path
import sys

# Change to True only when this kernel needs the section's packages installed.
INSTALL_PACKAGES = False

SETUP_HERE = Path.cwd().resolve()
SETUP_LAB = SETUP_HERE if (SETUP_HERE / "image_lab.py").is_file() else SETUP_HERE / "05_Jupyter"
REQUIREMENTS = SETUP_LAB / "requirements.txt"
if not REQUIREMENTS.is_file():
    raise FileNotFoundError("Start this kernel in the repository root or 05_Jupyter folder with requirements.txt present.")
print("Selected kernel Python:", sys.executable)

if INSTALL_PACKAGES:
    from IPython import get_ipython

    shell = get_ipython()
    if shell is None:
        raise RuntimeError("Run this setup cell in the notebook's IPython Python kernel.")
    # Equivalent to %pip install -r "{REQUIREMENTS}"; targets the selected kernel.
    shell.run_line_magic("pip", f'install -r "{REQUIREMENTS}"')
    print("Check pip output for errors. After success, set INSTALL_PACKAGES = False, restart the kernel, and run from the top.")
else:
    print("Installation skipped. If imports fail, set INSTALL_PACKAGES = True and rerun this cell first.")


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "05_Jupyter":
    ROOT = ROOT.parent
if not (ROOT / "05_Jupyter" / "notebook_tools.py").is_file():
    raise RuntimeError("Open this notebook from the repository root or 05_Jupyter directory.")
sys.path.insert(0, str(ROOT / "05_Jupyter"))
print("Repository found:", ROOT.name)

## 1. Define the experiment before training

Use scikit-learn's bundled 1,797-row digits dataset: 8×8 grayscale handwritten digits with ten classes. Its source is the UCI Optical Recognition of Handwritten Digits dataset, via scikit-learn's bundled copy. The original collection involved multiple writers; this simple stratified random split does **not** establish writer-independent generalization. No external dataset is downloaded.

We scale pixel values from 0..16 to 0..1 with a fixed divisor, then use a stratified holdout with a fixed seed. Because scaling has no learned statistics, it does not leak test information. Keep the holdout out of model selection; use a separate validation split for tuning.

In [ ]:
import json
import platform
from time import perf_counter
import numpy as np
import pandas as pd
import sklearn
import matplotlib
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay
from notebook_tools import save_metrics

SEED = 42
PARAMETERS = {"C": 1.0, "max_iter": 300, "solver": "lbfgs", "random_state": SEED}
digits = load_digits()
X_train, X_test, y_train, y_test = train_test_split(
    digits.data.astype(np.float64) / 16.0, digits.target,
    test_size=0.25, random_state=SEED, stratify=digits.target,
)
print("Train:", X_train.shape, "Test:", X_test.shape)
assert X_train.shape == (1347, 64) and X_test.shape == (450, 64)
assert set(y_train) == set(range(10)) == set(y_test)

## 2. Measure a baseline

Fit time and prediction time are separate wall-clock measurements. Runtime varies by machine, load, thread settings and library versions. These are not GPU or production latency benchmarks. A convergence warning means the optimization may need more iterations; investigate it rather than suppressing it.

In [ ]:
model = LogisticRegression(**PARAMETERS)
start = perf_counter()
model.fit(X_train, y_train)
fit_seconds = perf_counter() - start
start = perf_counter()
predicted = model.predict(X_test)
predict_seconds = perf_counter() - start
metrics = {
    "accuracy": float(accuracy_score(y_test, predicted)),
    "macro_f1": float(f1_score(y_test, predicted, average="macro")),
    "fit_seconds": fit_seconds,
    "predict_seconds": predict_seconds,
    "train_samples": len(y_train),
    "test_samples": len(y_test),
}
pd.DataFrame([metrics])

**Checkpoint:** expect accuracy around 0.95 or higher for this particular split, not a universal promise. Check macro-F1 too: it gives every class equal weight. Neither metric establishes robustness to new cameras, backgrounds, people or handwriting styles.

In [ ]:
OUTPUT = ROOT / "05_Jupyter" / "lab_outputs" / "experiment_report"
OUTPUT.mkdir(parents=True, exist_ok=True)
clean_metrics = save_metrics(metrics, OUTPUT)
for filename in ("confusion_matrix.png", "report.md"):
    if (OUTPUT / filename).is_symlink():
        raise ValueError("Refusing to overwrite a symlink output.")
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(y_test, predicted, labels=np.arange(10), cmap="Blues", colorbar=False, ax=ax)
ax.set_title("Digits baseline: held-out predictions")
fig.tight_layout()
fig.savefig(OUTPUT / "confusion_matrix.png", dpi=140)
plt.show()
plt.close(fig)

## 3. Assemble a report from explicit, safe fields

Export only validated finite numeric metrics and a short allowlist of package versions, fixed provenance and model parameters. Do not dump environment variables, credentials, filesystem listings or raw records. This report intentionally excludes serialized models. It can be reviewed locally before any sharing.

In [ ]:
versions = {
    "Python": platform.python_version(), "NumPy": np.__version__,
    "pandas": pd.__version__, "scikit-learn": sklearn.__version__, "Matplotlib": matplotlib.__version__,
}
report = f"""# Digits logistic-regression experiment

## Question and dataset
Can a small linear classifier distinguish ten digit classes in the bundled digits dataset?
Source: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html
Original source: https://archive.ics.uci.edu/dataset/80/optical+recognition+of+handwritten+digits
Provenance: scikit-learn bundled load_digits(), 1,797 images, 8×8 pixels, labels 0–9.
Pixel preprocessing: fixed division by 16.0; no learned preprocessing.

## Reproduction
Split: stratified train_test_split, 25% test, random_state={SEED}.
Training rows: {len(y_train)}; test rows: {len(y_test)}; seed: {SEED}.
Model: sklearn.linear_model.LogisticRegression.
Explicit parameters: {json.dumps(PARAMETERS, sort_keys=True)}.
Other parameters: library defaults for the versions recorded below.
Iterations used: {int(model.n_iter_.max())}.
Versions: {json.dumps(versions, sort_keys=True)}.
Rerun this notebook from the repository root or 05_Jupyter.

## Results
Accuracy: {clean_metrics['accuracy']:.4f}.
Macro-F1: {clean_metrics['macro_f1']:.4f}.
Fit wall time: {fit_seconds:.4f} seconds.
Prediction wall time for {len(y_test)} rows: {predict_seconds:.6f} seconds.

![Held-out confusion matrix](confusion_matrix.png)

Machine-readable results: [JSON](metrics.json), [CSV](metrics.csv).

## Interpretation and limits
This is one seeded random holdout, not cross-validation or a confidence interval.
Rows are split without writer groups; do not claim writer-independent performance.
The test set must not become a tuning set. Use separate validation data for model selection.
Tiny clean images differ from real photos. Accuracy does not establish deployment readiness.
Runtime is machine- and load-dependent, not a controlled hardware comparison.
Recorded versions and seed aid reproduction but do not guarantee bit-identical results everywhere.

## Next experiment
Compare two regularization strengths using a validation split within the training data.
Freeze the choice before evaluating once on the held-out test set.
"""
(OUTPUT / "report.md").write_text(report, encoding="utf-8")
print(report)

In [ ]:
# Verify exports round-trip and contain only the documented metrics.
loaded = json.loads((OUTPUT / "metrics.json").read_text(encoding="utf-8"))
assert loaded == clean_metrics
csv = pd.read_csv(OUTPUT / "metrics.csv")
assert len(csv) == 1 and set(csv.columns) == set(clean_metrics)
assert (OUTPUT / "confusion_matrix.png").stat().st_size > 1000
assert "## Interpretation and limits" in (OUTPUT / "report.md").read_text()
print("Verified report.md, metrics.json, metrics.csv and confusion_matrix.png")
print("Output directory:", OUTPUT.relative_to(ROOT))

## Practice, troubleshooting and cleanup

- Inspect the confusion matrix and describe one common confusion without inventing a cause. Add a manually reviewed note to a copy of the report; rerunning replaces the generated report.
- Create a training/validation split to compare regularization, then record the final held-out result. Do not select a model by repeatedly inspecting test accuracy.
- Add balanced accuracy by deliberately extending the helper's metric allowlist and tests. Explain why a strict schema protects an export from unintended data.
- Missing package: select the prepared kernel in [setup](README.md). Convergence warning: increase `max_iter` and record it. Permission error: check that this checkout is writable. Export validation error: inspect metrics for NaN or an unexpected field; do not disable validation.

**Cleanup:** generated files are ignored by Git but remain on disk. The optional cell below removes only this lab's four named artifacts. Clear notebook outputs before sharing and stop hosted compute when finished.

**Next:** [05k: Forecast Bike Demand](05k_Forecast_Bike_Demand.ipynb).

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    for filename in ("report.md", "metrics.json", "metrics.csv", "confusion_matrix.png"):
        (OUTPUT / filename).unlink(missing_ok=True)
    print("Removed only this lab's generated files.")
else:
    print("Outputs kept for review. Set CLEAN_UP=True only when finished.")